In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
import_folder = reports_folder.parent / "import"
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
import re
import numpy as np

from edc_appointment.models import Appointment
from edc_transfer.constants import TRANSFERRED
from clinicedc_constants import ADMINISTRATIVE_WITHDRAWAL, CONSENT_WITHDRAWAL, LTFU, NULL_STRING, YES
from meta_subject.models import StudyMedication, SubjectVisitMissed, SubjectVisit, FollowupExamination
from django_pandas.io import read_frame
from edc_model_to_dataframe.read_frame_edc import read_frame_edc
from django.conf import settings
from edc_registration.models import RegisteredSubject
from edc_visit_tracking.constants import MISSED_VISIT, UNSCHEDULED, SCHEDULED
from clinicedc_constants import OTHER
from meta_prn.models import EndOfStudy, OffSchedule, OffStudyMedication



In [ ]:
import re
import uuid
from decimal import Decimal

import numpy as np
import pandas as pd

STATA_RESERVED = {
    "_all", "_b", "byte", "_coef", "_cons", "double", "float", "if", "in", "int",
    "long", "_n", "_N", "_pi", "_pred", "_rc", "_skip", "strL", "using", "with",
}


def stata_name(name: str, seen: set) -> str:
    n = re.sub(r"\W", "_", str(name))
    if not re.match(r"[A-Za-z_]", n):
        n = f"_{n}"
    n = n[:32]
    if n in STATA_RESERVED:
        n = f"{n}_"[:32]
    base, i = n, 1
    while n.lower() in seen:
        suffix = f"_{i}"
        n = base[: 32 - len(suffix)] + suffix
        i += 1
    seen.add(n.lower())
    return n


def prepare_for_stata(df: pd.DataFrame, tz: str = "Africa/Dar_es_Salaam"):
    df = df.copy()
    convert_dates, convert_strl, labels = {}, [], {}

    # names: <=32 chars, valid identifiers, unique case-insensitively
    seen = set()
    rename = {c: stata_name(c, seen) for c in df.columns}
    labels = {new: str(old)[:80] for old, new in rename.items() if new != old}
    df = df.rename(columns=rename)

    for c in df.columns:
        s = df[c]

        # tz-aware datetimes -> local wall clock, naive (Stata has no tz)
        if isinstance(s.dtype, pd.DatetimeTZDtype):
            df[c] = s.dt.tz_convert(tz).dt.tz_localize(None)
            convert_dates[c] = "tc"
        elif pd.api.types.is_datetime64_any_dtype(s):
            convert_dates[c] = "tc"

        # timedeltas aren't supported -> days as float
        elif pd.api.types.is_timedelta64_dtype(s):
            df[c] = s.dt.total_seconds() / 86400

        # nullable ints / bools -> int if no NA, else float (NA -> Stata ".")
        elif isinstance(s.dtype, (pd.Int8Dtype, pd.Int16Dtype, pd.Int32Dtype,
                                  pd.Int64Dtype, pd.BooleanDtype)) or s.dtype == bool:
            df[c] = s.astype("float64") if s.isna().any() else s.astype("int64" if s.dtype != bool and not isinstance(s.dtype, pd.BooleanDtype) else "int8")

        elif isinstance(s.dtype, pd.CategoricalDtype):
            df[c] = s.cat.rename_categories(lambda x: str(x))

        # object / string columns
        elif s.dtype == object or isinstance(s.dtype, pd.StringDtype):
            non_null = s.dropna()
            sample = non_null.iloc[0] if len(non_null) else None

            if isinstance(sample, (pd.Timestamp,)) or type(sample).__name__ == "date":
                dt = pd.to_datetime(s)
                df[c] = dt.dt.tz_convert(tz).dt.tz_localize(None) if dt.dt.tz is not None else dt
                convert_dates[c] = "td" if type(sample).__name__ == "date" else "tc"
            elif isinstance(sample, Decimal):
                df[c] = pd.to_numeric(s, errors="coerce").astype("float64")
            else:
                # strings: UUIDs, mixed types, NA -> "" (Stata strings have no missing)
                df[c] = s.map(lambda v: "" if pd.isna(v) else str(v))
                if df[c].str.len().max() > 2045:
                    convert_strl.append(c)

    return df, dict(convert_dates=convert_dates, convert_strl=convert_strl,
                    variable_labels=labels)




In [ ]:
# tokens

IMP_TOKEN = re.compile(r"""
      (?<![a-z])[ij]mp(?![a-z])                # IMP, Imp, imp_complaints, JMP typo
    | toleratE?
    | side[\s_-]*effect
    | pill[\s_-]*burden
    | \b(?:drug|pill|tablet|medic\w*)s?\b[^;,]{0,30}?\b(?:sick|unwell|bad|ill)\b
    | \b(?:sick|unwell|bad|ill)\b[^;,]{0,30}?\b(?:drug|pill|tablet|medic)
""", re.I | re.X)

WITHDRAWAL_TOKEN = re.compile(r"""
      (?<!re)(?<!re-)(?<!re\s)consen           # consent, consen, consentient; not reconsent / re-consent
    | withdr[ae]w                              # withdraw, withdrawl, withdrawal, withdrawn, withdrew
    | \bopt(?:ed|s|ing)?[\s_-]*out
    | \bnot\s+willing
    | \bunwilling
    | \bnot\s+ready\s+to\s+continue
    | \b(?:don\W{0,2}t|do\s+not|doesn\W{0,2}t|does\s+not)\s+want\s+to\s+continue
""", re.I | re.X)

# OTHER_TOKEN = re.compile(rf"(?<![^,;\s]){re.escape(OTHER)}(?![^,;\s])")

OTHER_TEXTS = [OTHER, "Other reason (specify below)"]

OTHER_TOKEN = re.compile(
    r"(?<![^,;\s])(?:"
    + "|".join(re.escape(t) for t in sorted(OTHER_TEXTS, key=len, reverse=True))
    + r")(?![^,;\s])"
)


In [ ]:
tz = settings.TIME_ZONE  # or the site's zone, e.g. "Africa/Dar_es_Salaam"

def to_local(s):
    s = pd.to_datetime(s)
    if s.dt.tz is None:          # naive values from Django are UTC
        s = s.dt.tz_localize("UTC")
    return s.dt.tz_convert(tz)


def replace_other(reasons, other_text):
    if pd.isna(reasons) or pd.isna(other_text) or not str(other_text).strip():
        return reasons
    text = str(other_text).strip()
    return OTHER_TOKEN.sub(lambda m: text, str(reasons))

In [ ]:
# dataframe from elsa to compare
df_elsa = (
    pd.read_excel(import_folder / "withdrawal_end_dates_ek_v3.xlsx")
    .rename(columns={
        "Off study reason": "offstudy_reason",
        "Expected off study reason": "expected_offstudy_reason",
        "Date of consent withdrawal": "reason_date",
        "Reason for decision": "explanation",
        "Comment": "source_comments",
        "Location of comment": "source_document"
    })
)

df_elsa.loc[df_elsa["offstudy_reason"]=="consent withdrawal", "offstudy_reason"] = CONSENT_WITHDRAWAL
df_elsa.loc[df_elsa["expected_offstudy_reason"]=="verified", "expected_offstudy_reason"] = CONSENT_WITHDRAWAL
df_elsa.loc[df_elsa["expected_offstudy_reason"]=="Administrative withdrawal", "expected_offstudy_reason"] = ADMINISTRATIVE_WITHDRAWAL
df_elsa.loc[df_elsa["expected_offstudy_reason"]=="moved out of catchment area", "expected_offstudy_reason"] = TRANSFERRED
df_elsa["reason_date"] = pd.to_datetime(to_local(df_elsa["reason_date"])).dt.tz_convert(tz)


In [ ]:
# registred subject dataframe
df_rs = read_frame(
    RegisteredSubject.objects.values("subject_identifier", "consent_datetime", "randomization_datetime", "gender", "dob").all(), verbose=False
)
df_rs["subject_identifier"] = df_rs["subject_identifier"].astype(str)
df_rs["consent_datetime"] = pd.to_datetime(to_local(df_rs["consent_datetime"]))
df_rs["randomization_datetime"] = pd.to_datetime(to_local(df_rs["randomization_datetime"]))
df_rs["dob"] = pd.to_datetime(to_local(df_rs["dob"]))

In [ ]:
# EoS dataframe
df_eos = read_frame_edc(
    EndOfStudy.objects.all(),
    drop_sys_columns=True,
    drop_action_item_columns=True,
    read_frame_verbose=True
).rename(columns={"offstudy_reason_id": "offstudy_reason", "other_offstudy_reason": "offstudy_reason_other"})

df_eos = df_eos[["subject_identifier", "offstudy_datetime", "offstudy_reason", "offstudy_reason_other"]].reset_index(drop=True)

df_eos["offstudy_reason"] = [
    replace_other(r, o)
    for r, o in zip(df_eos["offstudy_reason"], df_eos["offstudy_reason_other"])
]
df_rs = df_rs.merge(df_eos[["subject_identifier", "offstudy_datetime", "offstudy_reason"]], on="subject_identifier", how="left")

In [ ]:
# df_eos.offstudy_reason.value_counts()

In [ ]:
# subject visit dataframe
df_visit = read_frame_edc(
    SubjectVisit.objects.all(),
    convert_visit_code_to_float=True,
    drop_sys_columns=True,
    drop_action_item_columns=True,
    read_frame_verbose=True
).rename(columns={"report_datetime": "visit_datetime"})
df_visit["visit_datetime"] = pd.to_datetime(to_local(df_visit["visit_datetime"]))
df_visit["last_alive_date"] = pd.to_datetime(to_local(df_visit["last_alive_date"]))
df_visit.loc[df_visit["reason"].isin(["Scheduled visit"]), "reason"] = SCHEDULED
df_visit.loc[df_visit["reason"].isin(["Unscheduled visit"]), "reason"] = UNSCHEDULED
df_visit.loc[df_visit["reason"].isin(["Missed visit"]), "reason"] = MISSED_VISIT

In [ ]:
# missed visits (subject visit missed) dataframe
df_missed_visit = read_frame_edc(
    SubjectVisitMissed.objects.all(),
    convert_visit_code_to_float=True,
    drop_sys_columns=True,
    drop_action_item_columns=True,
    read_frame_verbose=True
).rename(columns={"appointment_datetime": "appt_datetime"})
df_missed_visit["report_datetime"] = pd.to_datetime(to_local(df_missed_visit["report_datetime"]))
df_missed_visit["visit_datetime"] = pd.to_datetime(to_local(df_missed_visit["visit_datetime"]))
df_missed_visit["contact_last_date"] = pd.to_datetime(to_local(df_missed_visit["contact_last_date"]))
df_missed_visit["appt_datetime"] = pd.to_datetime(to_local(df_missed_visit["appt_datetime"]))

df_missed_visit["missed_reasons"] = [
    replace_other(r, o)
    for r, o in zip(df_missed_visit["missed_reasons"], df_missed_visit["missed_reasons_other"])
]

df_missed_visit = df_missed_visit.sort_values(["subject_identifier", "visit_code"])

In [ ]:
# clinical followup CRF dataframe
# only keep one record per subject that matches the WITHDRAWAL_TOKEN in comments

# after review of the grep, exclude these as false positives
false_positives = ["105-30-0209-1", "105-10-0113-7", "105-10-0008-9", "105-10-0130-1"]

df_followup = read_frame_edc(
    FollowupExamination.objects.all(),
    convert_visit_code_to_float=True,
    drop_sys_columns=True,
    drop_action_item_columns=True,
    read_frame_verbose=True
)
df_followup["subject_identifier"] = df_followup["subject_identifier"].astype(str)
df_followup["visit_datetime"] = pd.to_datetime(to_local(df_followup["visit_datetime"]))

s = df_followup["comment"].fillna("")
is_withdrawal = s.str.contains(WITHDRAWAL_TOKEN)
# is_imp = s.str.contains(IMP)

df_followup["withdrawal_signal"] = np.select(
    [is_withdrawal], ["withdrawal"], default=None
)
df_followup["withdrawal_signal"] = np.where(df_followup["visit_code"].isin([1360.0, 1480.0, 1480.1]), np.nan, df_followup["withdrawal_signal"])

df_followup = (
    df_followup[(df_followup["withdrawal_signal"]=="withdrawal") & ~(df_followup["subject_identifier"].isin(false_positives))][["subject_identifier", "visit_datetime", "comment"]]
    .rename(columns={"comment": "exam_withdrawal_comment", "visit_datetime": "exam_withdrawal_datetime"})
    .sort_values(["subject_identifier", "exam_withdrawal_datetime"])
    .drop_duplicates(subset=["subject_identifier"], keep="last")
    .reset_index(drop=True)
)

# merge with df_rs, fields are prfixed with `exam_withdrawal_`: exam_withdrawal_datetime, exam_withdrawal_comment
df_rs = df_rs.merge(df_followup, on='subject_identifier', how="left")
len(df_rs)


In [ ]:
# add consent_withdrawal_datetime
# TODO: is this superceded below with more aggressive WITHDRAW_TOKEN and IMP_TOKEN?

# WITHDRAW_TOKEN = r"\bconsent|withdr[ae]w\w*|s+consen"

has_consent = df_missed_visit["missed_reasons"].str.contains(
    WITHDRAWAL_TOKEN, regex=True, na=False
)
first_consent_dt = (
    df_missed_visit.loc[has_consent]
    .groupby("subject_identifier")["visit_datetime"]
    .min()
)
df_missed_visit["consent_withdrawal_datetime"] = (
    df_missed_visit["subject_identifier"].map(first_consent_dt)
)

In [ ]:
# add date of last contact to master patient list
# last_contact_datetime
last_contact = df_missed_visit.groupby(["subject_identifier"])["contact_last_date"].max()
df_rs = df_rs.join(last_contact, on="subject_identifier").rename(columns={"contact_last_date": "last_contact_datetime"})

In [ ]:
# add date last_seen
# last_seen_datetime
last_attended_visit = df_visit[~df_visit["reason"].isin([MISSED_VISIT])].groupby(["subject_identifier"])["visit_datetime"].max()
df_rs = df_rs.join(last_attended_visit, on="subject_identifier").rename(columns={"visit_datetime": "last_seen_datetime"})

In [ ]:
# add date last_seen
# last_visit_code
last_visit_code = df_visit[~df_visit["reason"].isin([MISSED_VISIT])].groupby(["subject_identifier"])["visit_code"].max().rename("last_visit_code")
df_rs = df_rs.join(last_visit_code, on="subject_identifier")

In [ ]:
# add date of last_missed_visit
# last_missed_visit_datetime
last_missed_visit = df_visit[df_visit["reason"].isin([MISSED_VISIT])].groupby(["subject_identifier"])["visit_datetime"].max()
df_rs = df_rs.join(last_missed_visit, on="subject_identifier").rename(columns={"visit_datetime": "last_missed_visit_datetime"})

In [ ]:
# add count of missed visits after last_seen_datetime
# missed_visits_after_last_seen
missed = df_missed_visit.merge(
    df_rs[["subject_identifier", "last_seen_datetime"]],
    on="subject_identifier",
    how="inner",
)
after = missed[missed["visit_datetime"] > missed["last_seen_datetime"]]
counts = after.groupby("subject_identifier").size()
df_rs["missed_visits_after_last_seen"] = (
    df_rs["subject_identifier"].map(counts).fillna(0).astype(int)
)


In [ ]:
# gather comments from missed visit and add to df_rs
# missed_visit_comments
comments = (
    after.sort_values(["subject_identifier", "visit_datetime"])
    .groupby("subject_identifier")["comment"]
    .agg(lambda s: "; ".join(s.dropna().astype(str).str.strip().loc[lambda x: x != ""]))
)
df_rs["missed_visit_comments"] = df_rs["subject_identifier"].map(comments).fillna("")

In [ ]:
# gather reasons from missed visit and add to df_rs
# missed_visit_reasons
# TODO: changed after to missed below
reasons = (
    missed.sort_values(["subject_identifier", "visit_datetime"])
    .groupby("subject_identifier")["missed_reasons"]
    .agg(lambda s: "; ".join(s.dropna().astype(str).str.strip().loc[lambda x: x != ""]))
)
df_rs["missed_visit_reasons"] = df_rs["subject_identifier"].map(reasons).fillna("")

In [ ]:
# add consent_withdrawal_datetime
map = (df_missed_visit[~df_missed_visit["consent_withdrawal_datetime"].isna()][["subject_identifier", "consent_withdrawal_datetime"]]
 .sort_values(["subject_identifier", "consent_withdrawal_datetime"])
 .drop_duplicates(subset=["subject_identifier", "consent_withdrawal_datetime"], keep="first")
 )

df_rs = df_rs.merge(
    map,
    on="subject_identifier",
    how="left"
)

df_rs.loc[df_rs["consent_withdrawal_datetime"] < df_rs["last_seen_datetime"], "consent_withdrawal_datetime"] = df_rs["last_seen_datetime"]

In [ ]:
cols = ["missed_visit_reasons", "missed_visit_comments", "exam_withdrawal_comment"]
df_rs["combined_comments"] = (
    df_rs[cols]
    .fillna("")
    .astype(str)
    .apply(lambda c: c.str.strip())
    .agg("|".join, axis=1)
)

In [ ]:
# take another pass at missed_visit_reasons

s = df_rs["combined_comments"].fillna("")
is_withdrawal = s.str.contains(WITHDRAWAL_TOKEN)
is_imp = s.str.contains(IMP_TOKEN)

df_rs["withdrawal_signal"] = np.select(
    [is_withdrawal, is_imp], ["consent_withdrawal", "imp_tolerability"], default=None
)
# df_rs[df_rs["missed_visit_reasons"].str.contains("consent")]
# df_rs[(~df_rs.missed_visit_reasons.isna()) & (df_rs.missed_visit_reasons!=NULL_STRING)].missed_visit_reasons

In [ ]:
cols = ["subject_identifier", "last_contact_datetime", "last_seen_datetime", "last_missed_visit_datetime", "missed_visits_after_last_seen", "exam_withdrawal_datetime", "missed_visit_comments", "missed_visit_reasons", "consent_withdrawal_datetime","withdrawal_signal", "expected_offstudy_reason", "reason_date", "Comments"]
df = df_rs.merge(df_elsa[["subject_identifier",  "expected_offstudy_reason", "reason_date", "Comments"]], on="subject_identifier", how="left") #[cols]


In [ ]:
df[df["subject_identifier"]=="105-40-0030-0"]

In [ ]:
df_export = (
    df[~(df.consent_withdrawal_datetime.isna()) | ~(df.withdrawal_signal.isna())][["subject_identifier", "last_visit_code", "last_contact_datetime", "last_seen_datetime", "last_missed_visit_datetime", "missed_visits_after_last_seen","exam_withdrawal_datetime", "consent_withdrawal_datetime", "withdrawal_signal", "expected_offstudy_reason", "reason_date", "Comments", "combined_comments"]]
    .sort_values("subject_identifier")
    .reset_index(drop=True)
)

df_export, kwargs = prepare_for_stata(df_export)
df_export.to_stata(path=analysis_folder / "eos_reasons.dta", version=118, write_index=False)


In [ ]:
df_rs[df_rs["subject_identifier"]=="105-30-0001-2"]

In [ ]:
df_elsa2 = (
    pd.read_excel(import_folder / "ltfu_end_dates_ek_v3.xlsx")
)


In [ ]:
df_elsa2

In [ ]:
def recalculate_number_of_days(df):
    df = df.sort_values(["subject_identifier", "visit_datetime"])
    refill_end = to_local(df["refill_end_datetime"]).dt.normalize()
    refill_start = to_local(df["refill_start_datetime"]).dt.normalize()
    df["number_of_days_recalc"] = (refill_end - refill_start).dt.days.fillna(0).astype("Int64")
    return df

def correct_refill_end_datetime(df):
    """Overwrite refill_end_datetime if refill to next == YES.

    The ModelForm was configured to calculate refill_end_datetime
    instead of accepting user input. This may have been applied
    some time after  study start.
    """
    df = df.sort_values(["subject_identifier", "timepoint_datetime"])
    # filter scheduled visits only
    next_visit = df[df["visit_code"] % 1 == 0].groupby("subject_identifier")["timepoint_datetime"].shift(-1)
    # don't overwrite of refill_to_next_visit is NO
    mask = df["refill_to_next_visit"].isin([YES]) & next_visit.notna()
    df.loc[mask, "refill_end_datetime"] = next_visit[mask]
    return df

In [ ]:
from edc_appointment.analytics import get_appointment_df
# 9 months off drug date
# last date supplied IMP

df_rx = read_frame_edc(
    StudyMedication.objects.all(),
    convert_visit_code_to_float=True,
    drop_sys_columns=True,
    drop_action_item_columns=True,
    read_frame_verbose=True
)


df_appt = (
    read_frame_edc(
        Appointment.objects.values(
            "subject_identifier",
            "visit_code",
            "visit_code_sequence",
            "appt_datetime",
            "timepoint_datetime",
            "appt_status",
            "appt_type__name",
            "appt_timing").filter(visit_schedule_name="visit_schedule", schedule_name="schedule"),
        convert_visit_code_to_float=True,
        drop_sys_columns=True,
        drop_action_item_columns=True,
        read_frame_verbose=False)
    .rename(columns={"appt_type_name": "appt_type"})
)

df_rx = (
    df_rx
    .merge(df_appt[["subject_identifier", "visit_code", "timepoint_datetime", "appt_datetime","appt_status","appt_type"]], on=["subject_identifier", "visit_code"], how="right")
    .drop(columns=["appointment_datetime", "site"]))

df_rx["report_datetime"] = pd.to_datetime(to_local(df_rx["report_datetime"]))
df_rx["refill_start_datetime"] = pd.to_datetime(to_local(df_rx["refill_start_datetime"]))
df_rx["refill_end_datetime"] = pd.to_datetime(to_local(df_rx["refill_end_datetime"]))
df_rx["timepoint_datetime"] = pd.to_datetime(to_local(df_rx["timepoint_datetime"]))
df_rx["appt_datetime"] = pd.to_datetime(to_local(df_rx["appt_datetime"]))
df_rx["visit_datetime"] = pd.to_datetime(to_local(df_rx["visit_datetime"]))

# corrections
df_rx.loc[df_rx["subject_identifier"]=="105-30-0227-3", "number_of_days"] = 0
df_rx.loc[(df_rx["subject_identifier"]=="105-20-0018-7") & (df_rx["visit_code"].isin([1000.0])), "refill_start_datetime"] = df_rx["report_datetime"]

# if refill to next == YES, overwrite refill_end_datetime

df_rx = df_rx.sort_values(["subject_identifier", "visit_code", "timepoint_datetime"])
df_rx["days_to_next_timepoint"] = (
    pd.to_timedelta(
        df_rx["timepoint_datetime"].dt.date
        .groupby(df_rx["subject_identifier"])
        .diff(-1).mul(-1)).dt.days
    ).astype("Int64").fillna(0)

df_rx = recalculate_number_of_days(df_rx)
df_rx = correct_refill_end_datetime(df_rx)
df_rx = recalculate_number_of_days(df_rx)

mask = ((df_rx["number_of_days_recalc"] > 96) & df_rx["refill_to_next_visit"].isin([YES])).fillna(False)
df_rx.loc[mask, "number_of_days_recalc"] = 96
df_rx["number_of_days_diff"] = abs(df_rx["number_of_days_recalc"] - df_rx["number_of_days"])

# days_to_next_appt
df_rx = df_rx.sort_values(["subject_identifier", "appt_datetime"])
df_rx["days_to_next_appt"] = (
    pd.to_timedelta(
        df_rx["appt_datetime"].dt.date
        .groupby(df_rx["subject_identifier"])
        .diff(-1).mul(-1)).dt.days
    ).astype("Int64").fillna(0)


# remove unattended appts
mask = ~df_rx["visit_datetime"].isna()
df_rx = df_rx[mask].copy().reset_index(drop=True)

# days_to_next_visit
df_rx = df_rx.sort_values(["subject_identifier", "visit_datetime"])
df_rx["days_to_next_visit"] = (
    pd.to_timedelta(
        df_rx["visit_datetime"].dt.date
        .groupby(df_rx["subject_identifier"])
        .diff(-1).mul(-1)).dt.days
    ).astype("Int64").fillna(0)



In [ ]:
df_rx[["subject_identifier", "visit_code", "visit_datetime", "report_datetime", "refill", "refill_to_next_visit", "number_of_days", "refill_start_datetime", "refill_end_datetime"]]

In [ ]:
df_rx = df_rx.sort_values(["subject_identifier", "visit_code"])
df_rx[df_rx["subject_identifier"]=="105-20-0018-7"][["subject_identifier", "visit_code", "appt_datetime", "visit_datetime", "number_of_days_diff", "number_of_days", "days_to_next_timepoint", "days_to_next_appt", "days_to_next_visit", "refill","refill_to_next_visit", "refill_start_datetime", "refill_end_datetime", "number_of_days_recalc"]]

In [ ]:
mask = df_rx["subject_identifier"]=="105-30-0065-7"
df_rx[mask][["subject_identifier", "visit_code", "timepoint_datetime", "days_to_next_timepoint", "days_to_next_appt", "days_to_next_visit"]]

In [ ]:
df_missed_visit = read_frame_edc(
    SubjectVisitMissed.objects.all(),
    convert_visit_code_to_float=True,
    drop_sys_columns=True,
    drop_action_item_columns=True,
    read_frame_verbose=True
)



In [ ]:
subject_identifiers = [
    "105-10-0054-3",
    "105-10-0162-4",
    "105-30-0317-2",
    "105-40-0019-3",
    "105-40-0030-0",
    "105-40-0034-2",
    "105-40-0051-6",
    "105-40-0282-7",
    "105-60-0056-3",
    "105-60-0121-5",
]
df_rs[df_rs["subject_identifier"].isin(subject_identifiers)]

In [ ]:
df_study_med = read_frame_edc(
    StudyMedication.objects.all(),
    convert_visit_code_to_float=True,
    drop_sys_columns=True,
    drop_action_item_columns=True,
    read_frame_verbose=True
)

df_offmeds = read_frame_edc(
    OffStudyMedication.objects.all(),
    convert_visit_code_to_float=True,
    drop_sys_columns=True,
    drop_action_item_columns=True,
    read_frame_verbose=True
)

df_off = read_frame_edc(
    OffSchedule.objects.all(),
    drop_action_item_columns=True,
    read_frame_verbose=True
)



In [ ]:

subject_identifier = "105-60-0060-5"

In [ ]:
df_rs[df_rs["subject_identifier"]==subject_identifier][["subject_identifier", "offstudy_datetime", "offstudy_reason", "last_contact_datetime", "last_seen_datetime", "last_visit_code", "last_missed_visit_datetime", "combined_comments", ]]

In [ ]:
df_rs[df_rs["subject_identifier"]==subject_identifier][["subject_identifier", "combined_comments", ]]

In [ ]:
df_study_med[df_study_med["subject_identifier"]==subject_identifier]

In [ ]:
df_offmeds[df_offmeds["subject_identifier"]==subject_identifier][["subject_identifier", "report_datetime", "stop_date", "last_dose_date", "reason", "reason_other", "comment"]]


In [ ]:
df_missed_visit[df_missed_visit["subject_identifier"]==subject_identifier][["subject_identifier", "report_datetime", "visit_code", "contact_last_date", "missed_reasons", "missed_reasons_other"]].sort_values("report_datetime")


In [ ]:
df_visit[df_visit["subject_identifier"]==subject_identifier][["subject_identifier", "visit_datetime", "visit_code", "reason"]].sort_values("visit_datetime")

In [ ]:
df_appt[df_appt["subject_identifier"]==subject_identifier][["subject_identifier", "appt_datetime", "visit_code", "appt_timing", "appt_status"]].sort_values("appt_datetime")


In [ ]:
df_off[df_off["subject_identifier"]==subject_identifier][["subject_identifier", "offschedule_datetime", "created"]]